## Production Inference Deployment with PyTorch

First thing to do:
- putting model in evaluation mode: 
    - turns off AutoGrad, computation is expensive
    - Affects activity of certain layers
        - **Dropout** is only active during trianing
        - **BatchNorm** only tracks running mean and variance during trianing

Putting model in Evaluation Mode:
```
model=MyModel();
model.load_state_dict(torch.laod(my_weights_file))

model.eval() or model.train(False)

model.train(my_training_flag)

model(my_inference_batch) send model and data


## What is TorchScript
- statically typed subset of Python for ML
- meant for consumption by PyTorch Just in Time compiler which perform run time optimizations
- preferred method for serializing your trained model and deploying it for production inference

to convert to torchscript
scripted_model=torch.jit.script(model)
sample_input = torch.rand(MY_INPUT_SIZE).unsqueeze(0)
traced_model = torch.jit.trace(mode., sample_input)

.script() vs .trace()
torch.jit.script():
- converts model by analyzing Python code
- preserves control flow(conditionals, loops etc)
- accommodates list, dict, tuple
- may not cover 100% of operators

torch.jit.trace():
- requires a sample input to trace the computation path
- does not preserve control flow
- works with just about any code

model.save('my_module.pt') saves both computation graph and learning weights in a single file

inference, model=torch.jit.load('my_module')
preidction_batch=model(input_batch)

## Using TorchScript Module in C++

cmake_minimum_required(VERSION 3.0 FATAL_ERROR)
project(custom_ops)

find_package(Torch REQUIRED)

add_executable(example-app example-app.cpp)
target_link_libraries(example-app "${TORCH_LIBRARIES}")
set_property(TARGET example-app PROPERTY CXX_STANDARD 14)

```
#include <torch/script.h>

torch::jit::script::Module module;
try {
    modeule=torch::jit::load(argv[1]);
}
catch(const c10::Error& e){
    return -1;
}

std::Vector<torch::jit::IValue> inputs;
inputs.push_back(torch::ones(){1,3,224,224}); //dummy input vector

at::Tensor outputs = module.forward(inputs).toTensor(); //get inference from model
``` 

Torchserve
- default handlers for common use cases, along with custom handlers for others
- Model versioning and ability to roll back to earlier version
- automatic batching individual inferences across HTTP requests
- Logging including common metrics, and ability to incorporate custom metrics
- Robust HTTPS APIS Management and Inference